# Giga Meter — Single School Deep Dive

Everything known about **one school**, with automatic flags for the anomalies
that matter when someone asks "what is going on at this school?".

Set `SCHOOL_ID` (or `SCHOOL_NAME_CONTAINS`) in Part 0 and run top to bottom.
Fleet comparisons are drawn from the same country/province, so every figure is
"versus its peers", not versus an absolute.

**Inputs** — the caches written by `download_data_01.ipynb`, plus ping:
| File | Contents |
|---|---|
| `<slug>_clean.parquet` | cleaned measurements |
| `<ISO3>_master_datapull.csv` | school master |
| `<iso3>_ping_daily_local.parquet` | ping per school-day (local dates) |
| `<iso3>_ping_hourly_local.parquet` | ping per school-hour (optional, for the day curve) |

Ping caches are pulled with `format_measurements.temp_ping_daily_query()`; see
the refresh snippet in Part 0.

---
## Part 0 — Parameters and load

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import sys, json, re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

set_up_dir = Path.cwd() / "helpers"
if str(set_up_dir) not in sys.path:
    sys.path.insert(0, str(set_up_dir))

from format_measurements import localise_dates
from giga_chart_style import (GIGA_BLUE, GIGA_GREY, GIGA_GOOD, GIGA_MODERATE,
                              GIGA_BAD, GIGA_PRIMARY, GIGA_SUPTITLE)

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)
print("\u2713 imports complete")

In [ ]:
# =============================================================================
# PARAMETERS — set the school here
# =============================================================================
COUNTRY_NAME = "South Africa"
COUNTRY_ISO3 = "ZAF"
CACHE_DIR    = Path.home() / f"Documents/Giga/Delivery/Countries/{COUNTRY_NAME}"

# Identify the school either way; SCHOOL_ID wins if both are set.
SCHOOL_ID             = "1175b3b4-4b2e-3143-8916-31f865dc3149"   # Bekizulu SSS
SCHOOL_NAME_CONTAINS  = None          # e.g. "BEKIZULU"

PEER_SCOPE = 'admin1'   # 'admin1' = compare against the province, None = country

# Service thresholds. NOTE: the project default in *_clean_params.json is
# thr_latency_ms = 100; 150 is used in the ZAF landscape work. Set explicitly so
# the choice is visible rather than inherited silently.
THR_DL, THR_UL, THR_LAT = 20, 10, 150

EXPECTED_PINGS_PER_DAY = 48    # 4/hr x 12 hrs (08:00-20:00), PER DEVICE
SAVE_FIGS  = True
OUTPUT_DIR = CACHE_DIR / "school_dives"
OUTPUT_DIR.mkdir(exist_ok=True)

def savefig(fig, name):
    if SAVE_FIGS:
        fig.savefig(OUTPUT_DIR / f"{name}.png", dpi=200, bbox_inches='tight')

print(f"cache:  {CACHE_DIR}")
print(f"output: {OUTPUT_DIR}")

In [ ]:
# =============================================================================
# LOAD
# =============================================================================
slug = COUNTRY_NAME.lower().replace(' ', '')
PARAMS = json.loads((CACHE_DIR / f"{slug}_clean_params.json").read_text())
TIMEZONE = PARAMS['country']['timezone']

m      = localise_dates(pd.read_parquet(CACHE_DIR / f"{slug}_clean.parquet"), timezone=TIMEZONE)
master = pd.read_csv(CACHE_DIR / f"{COUNTRY_ISO3}_master_datapull.csv", low_memory=False)

def _try(path):
    return pd.read_parquet(path) if path.exists() else None

ping   = _try(CACHE_DIR / f"{COUNTRY_ISO3.lower()}_ping_daily_local.parquet")
hourly = _try(CACHE_DIR / f"{COUNTRY_ISO3.lower()}_ping_hourly_local.parquet")
if ping is not None:
    ping['date'] = pd.to_datetime(ping['date'])
print(f"measurements {len(m):,} \u00b7 master {len(master):,} \u00b7 "
      f"ping {'none' if ping is None else f'{len(ping):,}'}")

# Refresh the ping caches (needs the Trino port-forward):
#   from load_measurements import get_trino_engine
#   import format_measurements as fm
#   q = fm.temp_ping_daily_query('ZA', TIMEZONE, since='2025-09-01')
#   pd.read_sql(q, get_trino_engine()).to_parquet(
#       CACHE_DIR / f"{COUNTRY_ISO3.lower()}_ping_daily_local.parquet", index=False)

In [ ]:
# =============================================================================
# RESOLVE THE SCHOOL + BUILD ITS PEER GROUP
# =============================================================================
if SCHOOL_ID:
    sid = SCHOOL_ID
else:
    hits = m[m.school_name.str.contains(SCHOOL_NAME_CONTAINS, case=False, na=False)]
    ids  = hits.school_id_giga.unique()
    if len(ids) == 0:
        raise ValueError(f"no school matching {SCHOOL_NAME_CONTAINS!r}")
    if len(ids) > 1:
        display(hits.groupby('school_id_giga').school_name.first().to_frame())
        raise ValueError(f"{len(ids)} schools match — set SCHOOL_ID")
    sid = ids[0]

sm = m[m.school_id_giga == sid].copy()
if sm.empty:
    raise ValueError(f"no measurements for {sid}")

ADMIN1 = sm.admin1.mode().iloc[0]
peers  = m[m.admin1 == ADMIN1] if PEER_SCOPE == 'admin1' else m
PEER_LABEL = ADMIN1 if PEER_SCOPE == 'admin1' else COUNTRY_NAME

# School-hours weekday frames — the basis for every performance figure
def school_hours(df):
    return df[(df.measurement_time_window == 'school_hours') & (df.is_weekday.astype(bool))]

sh_school = school_hours(sm)
sh_peers  = school_hours(peers)

print(f"school:  {sm.school_name.iloc[0]}")
print(f"peers:   {PEER_LABEL} \u2014 {peers.school_id_giga.nunique()} schools")

---
## 1 — Identity card

In [ ]:
# =============================================================================
# 1 — WHO IS THIS SCHOOL
# =============================================================================
mrow = master[master.school_id_giga == sid]
mrow = mrow.iloc[0] if len(mrow) else None

card = {
    'school_name':    sm.school_name.iloc[0],
    'school_id_giga': sid,
    'school_id_govt': sm.school_id_govt.iloc[0],
    'province':       ADMIN1,
    'district':       sm.admin2.mode().iloc[0] if sm.admin2.notna().any() else None,
    'education_level': sm.education_level.mode().iloc[0] if sm.education_level.notna().any() else None,
    'area_type':      sm.school_area_type.mode().iloc[0] if sm.school_area_type.notna().any() else None,
    'latitude':       sm.latitude.median(),
    'longitude':      sm.longitude.median(),
}
if mrow is not None:
    card.update({
        'learners':          mrow.get('num_students'),
        'teachers':          mrow.get('num_teachers'),
        'connectivity_govt': mrow.get('connectivity_govt'),
        'type_govt':         mrow.get('connectivity_type_govt'),
        'contracted_mbps':   mrow.get('download_speed_contracted'),
    })
card.update({
    'measurements':   len(sm),
    'first_measured': sm.date.min().date(),
    'last_measured':  sm.date.max().date(),
    'test_days':      sm.date.dt.date.nunique(),
    'app_versions':   ', '.join(sorted(sm.app_version.dropna().unique())),
})
display(pd.Series(card, name='').to_frame())

---
## 2 — Time series

In [ ]:
# =============================================================================
# 2 — SPEED, LATENCY AND PING OVER TIME
# =============================================================================
# Daily medians for the school, with its peer-group median behind it, so a dip
# that is happening everywhere is not read as a fault at this school.
sd = (sh_school.assign(d=sh_school.date.dt.date)
      .groupby('d').agg(dl=('download_speed','median'), ul=('upload_speed','median'),
                        lat=('latency','median'), n=('download_speed','size')))
sd.index = pd.to_datetime(sd.index)

pd_peers = (sh_peers.assign(d=sh_peers.date.dt.date)
            .groupby('d').agg(dl=('download_speed','median'), ul=('upload_speed','median'),
                              lat=('latency','median')))
pd_peers.index = pd.to_datetime(pd_peers.index)

has_ping = ping is not None and (ping.school_id_giga == sid).any()
nrows = 4 if has_ping else 3
fig, axes = plt.subplots(nrows, 1, figsize=(12, 2.6*nrows), sharex=True)

for ax, col, thr, label, higher_better in [
        (axes[0], 'dl',  THR_DL,  'Download (Mbps)', True),
        (axes[1], 'ul',  THR_UL,  'Upload (Mbps)',   True),
        (axes[2], 'lat', THR_LAT, 'Latency (ms)',    False)]:
    ax.plot(pd_peers.index, pd_peers[col], color=GIGA_GREY[300], lw=1,
            label=f'{PEER_LABEL} median', zorder=2)
    ok = (sd[col] >= thr) if higher_better else (sd[col] <= thr)
    ax.scatter(sd.index, sd[col], s=14, color=np.where(ok, GIGA_GOOD, GIGA_BAD), zorder=4)
    ax.plot(sd.index, sd[col], color=GIGA_BLUE, lw=.8, alpha=.5, zorder=3)
    ax.axhline(thr, color='black', lw=1, ls='--', zorder=1)
    ax.set_ylabel(label)
    ax.legend(frameon=False, fontsize=8, loc='upper left')
    for s in ('top','right'): ax.spines[s].set_visible(False)
    ax.grid(color=GIGA_GREY[200], lw=.6); ax.set_axisbelow(True)

if has_ping:
    pg = ping[ping.school_id_giga == sid].set_index('date').sort_index()
    pg['pct_ok'] = 100 * pg.n_connected / pg.n_pings
    ax = axes[3]
    ax.bar(pg.index, pg.pct_ok, color=np.where(pg.pct_ok >= 90, GIGA_GOOD, GIGA_BAD), width=1.0)
    ax.set_ylabel('Ping success (%)'); ax.set_ylim(0, 100)
    for s in ('top','right'): ax.spines[s].set_visible(False)
    ax.grid(axis='y', color=GIGA_GREY[200], lw=.6); ax.set_axisbelow(True)

axes[0].set_title(f"{card['school_name']} \u2014 daily medians vs {PEER_LABEL}",
                  fontproperties=GIGA_SUPTITLE, loc='left', pad=12)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout(); savefig(fig, f"dive_{sid[:8]}_timeseries"); plt.show()

In [ ]:
# =============================================================================
# 2b — WHAT THE SCHOOL CONNECTS THROUGH, OVER TIME
# =============================================================================
# ISP and network name by month. A school switching between a fixed line and a
# phone hotspot shows up here as alternating rows.
HOTSPOT_RE = r'iphone|android|galaxy|samsung|redmi|hotspot|mifi'
sm['month'] = sm.date.dt.to_period('M')
sm['is_hotspot'] = sm.detected_wifi_ssid.str.contains(HOTSPOT_RE, case=False, na=False)

by_month = sm.groupby('month').agg(
    tests=('download_speed','size'),
    dl=('download_speed','median'), ul=('upload_speed','median'),
    isps=('isp_mapped', lambda s: ', '.join(sorted(s.dropna().unique()))),
    ssids=('detected_wifi_ssid', lambda s: ', '.join(sorted(s.dropna().unique())[:3])),
    pct_hotspot=('is_hotspot', lambda s: round(100*s.mean()) if s.notna().any() else np.nan))
display(by_month.round(1))

if sm.detected_wifi_ssid.notna().any():
    print("network names seen (top 10):")
    display(sm.detected_wifi_ssid.value_counts().head(10).to_frame('measurements'))

---
## 3 — Anomaly flags

In [ ]:
# =============================================================================
# 3 — AUTOMATIC FLAGS
# =============================================================================
# Each check compares the school against its peer group and raises a flag with
# the evidence attached. Thresholds are stated inline so a flag can be argued
# with rather than taken on trust.
flags = []
def flag(level, title, detail):
    flags.append({'level': level, 'flag': title, 'evidence': detail})

# --- performance ------------------------------------------------------------
dl_s, ul_s, lat_s = sh_school.download_speed.median(), sh_school.upload_speed.median(), sh_school.latency.median()
peer_school_med = sh_peers.groupby('school_id_giga').download_speed.median()
pctile = 100 * (peer_school_med < dl_s).mean()

if dl_s < THR_DL:
    flag('service', 'Download below threshold',
         f"{dl_s:.1f} Mbps vs {THR_DL} target \u00b7 {pctile:.0f}th percentile in {PEER_LABEL}")
if ul_s < THR_UL:
    flag('service', 'Upload below threshold', f"{ul_s:.1f} Mbps vs {THR_UL} target")
if lat_s > THR_LAT:
    flag('service', 'Latency above threshold', f"{lat_s:.0f} ms vs {THR_LAT} target")

# --- consistency: weekly coefficient of variation ---------------------------
sh_school = sh_school.copy()
sh_school['week'] = sh_school.timestamplocal.dt.tz_localize(None).dt.to_period('W-SUN').dt.start_time
wk = sh_school.groupby('week').download_speed.agg(['size','mean','std'])
wk = wk[wk['size'] >= 5]
if len(wk) >= 3:
    cov = (100 * wk['std'] / wk['mean']).median()
    peer_cov = []
    _p = sh_peers.copy()
    _p['week'] = _p.timestamplocal.dt.tz_localize(None).dt.to_period('W-SUN').dt.start_time
    _g = _p.groupby(['school_id_giga','week']).download_speed.agg(['size','mean','std'])
    _g = _g[_g['size'] >= 5]
    peer_cov = (100*_g['std']/_g['mean']).groupby('school_id_giga').median()
    if cov > peer_cov.quantile(.75):
        flag('consistency', 'Speed swings more than most peers',
             f"weekly CoV {cov:.0f}% vs {PEER_LABEL} median {peer_cov.median():.0f}% "
             f"(top quartile above {peer_cov.quantile(.75):.0f}%)")

# --- ping: coverage and success ---------------------------------------------
if has_ping:
    pg = ping[ping.school_id_giga == sid]
    dev = 1
    if hourly is not None:
        _hs = hourly[hourly.school_id_giga == sid]
        if len(_hs):
            dev = _hs.groupby('d').devices.max().median()
    recorded, responded = pg.n_pings.sum(), pg.n_connected.sum()
    expected = EXPECTED_PINGS_PER_DAY * dev * pg.date.nunique()
    coverage, success = 100*recorded/expected, 100*responded/recorded

    _pp = ping[ping.school_id_giga.isin(set(peers.school_id_giga))]
    _ps = (_pp.groupby('school_id_giga')
              .apply(lambda g: 100*g.n_connected.sum()/g.n_pings.sum(), include_groups=False))

    if success < _ps.quantile(.10):
        flag('availability', 'Connection rarely responds when tested',
             f"{success:.0f}% of {recorded:,} ping checks succeeded \u00b7 "
             f"{PEER_LABEL} median {_ps.median():.0f}%, bottom decile below {_ps.quantile(.10):.0f}%")
    if coverage < 15:
        flag('deployment', 'Device rarely running',
             f"only {coverage:.0f}% of the {expected:,.0f} expected ping checks ran")

    # the signature worth naming: fast when up, but seldom up
    if success < 50 and dl_s >= THR_DL:
        flag('availability', 'Fast when connected, but seldom connected',
             f"download {dl_s:.1f} Mbps (meets {THR_DL}) yet only {success:.0f}% of ping checks succeed "
             "\u2014 an intermittent link, not a slow one")

# --- dependence on a phone hotspot ------------------------------------------
if sm.detected_wifi_ssid.notna().any():
    _d = sm[sm.detected_wifi_ssid.notna()].assign(day=lambda d: d.date.dt.date)
    hs_days = _d[_d.is_hotspot].day.nunique(); all_days = _d.day.nunique()
    if all_days and 100*hs_days/all_days >= 50:
        top = _d[_d.is_hotspot].detected_wifi_ssid.mode()
        flag('infrastructure', 'Runs on a phone hotspot',
             f"{100*hs_days/all_days:.0f}% of days with a network name "
             f"({hs_days}/{all_days})" + (f" \u00b7 e.g. {top.iloc[0]}" if len(top) else ""))

# --- multiple providers ------------------------------------------------------
n_isp = sm.isp_mapped.nunique()
if n_isp >= 3:
    flag('infrastructure', 'Several providers seen',
         f"{n_isp} ISPs: {', '.join(sorted(sm.isp_mapped.dropna().unique())[:5])}")

# --- thin evidence -----------------------------------------------------------
if sm.date.dt.date.nunique() < 10:
    flag('data', 'Thin measurement record',
         f"only {sm.date.dt.date.nunique()} test days \u2014 treat every figure above as indicative")
gap = sm.date.sort_values().diff().dt.days.max()
if pd.notna(gap) and gap >= 60:
    flag('data', 'Long gap in measurement',
         f"{gap:.0f} days without a test")

# --- report ------------------------------------------------------------------
print(f"{card['school_name']} \u2014 {len(flags)} flag(s)\n")
if flags:
    display(pd.DataFrame(flags).set_index('level'))
else:
    print("No anomalies against the peer group.")

---
## Method notes

- **Peer group** is the school's own province by default (`PEER_SCOPE`), so
  flags read as "unusual for this province", not "unusual in the abstract".
- **Ping checks measure device running time, not connection uptime.** Missing
  checks mean the app was not running: failed checks *are* recorded
  (`is_connected = false`), so an absent row cannot be an outage. Coverage and
  success are therefore reported separately and never combined into "uptime".
- **`EXPECTED_PINGS_PER_DAY` is per device.** Schools running two devices are
  scaled accordingly, using the hourly cache where available.
- **Wireless fields need app 2.0.2+.** A school on an older build has no
  network name, so the hotspot check silently cannot fire — absence of the flag
  is not evidence of absence.
- **`detected_wifi_*` describes the measuring laptop**, not school network
  equipment.
- **Latency threshold**: set in Part 0 rather than inherited, because the
  project default (100 ms) and the ZAF landscape work (150 ms) differ.